# Tropycal quickstart — one Atlantic season

Fetch tropical-cyclone **best tracks** for the 2005 North Atlantic season from HURDAT2 through the `earthlens` Tropycal backend and plot them coloured by Saffir-Simpson category.

Tropycal is a `vector` backend: `download()` returns a pyramids `FeatureCollection` (a `geopandas.GeoDataFrame`). The default geometry is one `Point` per 6-hourly fix.

> **First-load cost.** The first query of a basin downloads and parses its whole best-track file (a few seconds for HURDAT, longer for IBTrACS). Needs `pip install earthlens[tropycal]`.

In [ ]:
from pathlib import Path

from earthlens import EarthLens

OUT_DIR = Path('tropycal_output')
OUT_DIR.mkdir(exist_ok=True)

## Query

`variables` selects the **basin** (here `north_atlantic`). The whole season's storms are filtered to the date window and bbox at the fix level. The result is written to `tropycal_output/` and returned in memory.

In [ ]:
# Live best-track query. Wrapped so the notebook stays safe under
# nbval-lax when run offline / without the [tropycal] extra.
fixes = None
try:
    fixes = EarthLens(
        variables=['north_atlantic'],
        data_source='tropycal',
        start='2005-08-01',
        end='2005-09-15',
        lat_lim=[18.0, 31.0],
        lon_lim=[-98.0, -80.0],
        source='hurdat',
        path=str(OUT_DIR),
    ).download(progress_bar=False)
    print(len(fixes), 'fixes')
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

## Inspect the fixes

In [ ]:
if fixes is not None and len(fixes):
    cols = ['storm_id', 'name', 'time', 'vmax_kt', 'category', 'geometry']
    display(fixes[cols].head())
    print('CRS:', fixes.crs)

## Plot the track fixes coloured by category

In [ ]:
if fixes is not None and len(fixes):
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 6))
    fixes.plot(ax=ax, column='category', cmap='YlOrRd', legend=True,
               markersize=25, edgecolor='k', linewidth=0.2)
    ax.set_title('North Atlantic best-track fixes, Aug-Sep 2005 (Gulf of Mexico)')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.show()